# 数据清洗：让脏数据变得干净可用
> 难度标签：初级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：01_数据预处理 | 源文件：数据清洗.py | 核心功能：重复值、异常值、格式不一致等常见数据质量问题的处理

## 概述

"Garbage in, garbage out"——数据科学界最经典的格言。不管你的模型多精妙，如果喂进去的数据是脏的，出来的结果也一定是垃圾。

数据清洗是整个 ML 流程中最耗时但最容易被教程忽略的环节。据统计，数据科学家 60-80% 的时间花在数据清洗和准备上。这个脚本模拟了真实场景中最常见的脏数据问题：重复行、异常值、格式不一致、空白字符，并演示了系统化的清洗流程。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
import pandas as pd

## 数学原理

### 1. IQR 异常值检测的统计学基础

**代码对应**：`detect_outliers_iqr(series, factor=1.5)` 函数实现了 IQR 方法。

四分位距（Interquartile Range）定义为：

$$\text{IQR} = Q_3 - Q_1$$

其中 $Q_1$ 为第 25 百分位数，$Q_3$ 为第 75 百分位数。异常值的判定边界为：

$$L = Q_1 - k \cdot \text{IQR}, \quad U = Q_3 + k \cdot \text{IQR}$$

**为什么默认 $k=1.5$？** 对于正态分布 $X \sim \mathcal{N}(\mu, \sigma^2)$：

$$Q_1 = \mu - 0.6745\sigma, \quad Q_3 = \mu + 0.6745\sigma, \quad \text{IQR} = 1.349\sigma$$

因此 $Q_1 - 1.5 \cdot \text{IQR} = \mu - 2.6985\sigma$，$Q_3 + 1.5 \cdot \text{IQR} = \mu + 2.6985\sigma$。

该范围覆盖正态分布的 99.3% 数据。超出范围的概率仅为：

$$P(|X - \mu| > 2.6985\sigma) \approx 0.7\%$$

$k=3$ 时边界为 $\mu \pm 4.047\sigma$，仅用于检测极端异常值（"far out"）。

**IQR 方法的优势**：$Q_1$ 和 $Q_3$ 的崩溃点（breakdown point）为 25%，即即使 25% 的数据被污染，这两个统计量仍然可靠。相比之下，均值的崩溃点为 0%——一个极端值就能改变均值。

### 2. Z-score 异常值检测与正态假设

**代码对应**：`np.abs(stats.zscore(data[col].dropna()))` 计算每个样本的 Z-score。

Z-score 定义为：

$$z_i = \frac{x_i - \bar{x}}{s}$$

其中 $\bar{x} = \frac{1}{n}\sum_{i=1}^n x_i$ 为样本均值，$s = \sqrt{\frac{1}{n-1}\sum_{i=1}^n (x_i - \bar{x})^2}$ 为样本标准差。

**正态假设下的阈值含义**：对于 $X \sim \mathcal{N}(\mu, \sigma^2)$：

$$P(|Z| > 3) = 2\Phi(-3) \approx 0.0027 \approx 0.27\%$$

即 1000 个样本中约 2.7 个会被标记为异常值。

**Z-score 的局限性**：

1. **均值和标准差本身受异常值影响**。一个极端值 $x_{\max}$ 会拉高 $\bar{x}$ 和 $s$，导致其他异常值的 Z-score 被"稀释"。改进方案：使用修正 Z-score（Modified Z-score）：

$$M_i = \frac{0.6745(x_i - \tilde{x})}{\text{MAD}}$$

其中 $\tilde{x}$ 为中位数，$\text{MAD} = \text{median}(|x_i - \tilde{x}|)$ 为中位绝对偏差。MAD 的崩溃点为 50%。

2. **非正态分布失效**。偏态分布中，远离均值的点可能是正常的。例如对数正态分布的尾部天然很长。

### 3. Winsorize 截断的损失函数视角

**代码对应**：`data[col] = data[col].clip(lower=lower, upper=upper)` 将值截断到 $[L, U]$。

截断操作可以写为：

$$\tilde{x}_i = \text{clip}(x_i, L, U) = \max(L, \min(x_i, U))$$

从损失函数的角度，Winsorize 等价于对异常值使用 Huber 损失的思想：

$$\rho_H(z) = \begin{cases} \frac{1}{2}z^2 & \text{if } |z| \leq k \\ k|z| - \frac{1}{2}k^2 & \text{if } |z| > k \end{cases}$$

Huber 损失对小残差使用平方损失（类似 MSE），对大残差使用线性损失（类似 MAE），从而降低异常值的影响。Winsorize 通过截断实现了类似的效果——极端值被"拉回"到合理边界。

### 4. 去重的集合论描述

**代码对应**：`data.duplicated()` 检测重复行，`data.drop_duplicates()` 删除。

设数据集为行向量的多重集合 $D = \{r_1, r_2, \ldots, r_N\}$。去重操作为：

$$D_{\text{unique}} = \{r \in D : \text{first\_occurrence}(r)\}$$

即保留每个唯一样本的首次出现。去重后 $|D_{\text{unique}}| \leq |D|$。

在代码中，`duplicated()` 对每一行计算是否与之前的行完全相同（逐列比较），时间复杂度为 $O(N \cdot d)$，其中 $d$ 为特征数。对于大规模数据，可用哈希表将平均复杂度降为 $O(N)$。

### 5. 异常值处理策略的统计影响

不同异常值处理方法对数据统计量的影响：

| 方法 | 均值 | 方差 | 样本量 | 适用场景 |
|------|------|------|--------|---------|
| 删除 | 不变 | 不变 | 减少 | 缺失少、数据充足 |
| Winsorize 截断 | 偏移 | 压缩 | 不变 | 保留样本量 |
| Z-score 过滤 | 不变 | 不变 | 减少 | 正态分布数据 |
| IQR 过滤 | 不变 | 不变 | 减少 | 偏态分布更鲁棒 |

### 6. 数据清洗流程的顺序依赖性

数据清洗的各步骤不是独立的，顺序会影响结果。推荐流程的数学性质：

1. **去重**：$\text{dedup}(D) \to D'$——先去重避免后续处理的计算浪费
2. **格式标准化**：$\text{normalize}(D') \to D''$——确保字符串比较的一致性
3. **异常值处理**：$\text{clip}(D'', L, U) \to D'''$——在缺失值填充前处理异常值
4. **缺失值处理**：$\text{impute}(D''') \to D^{\text{clean}}$——最后填充缺失值

如果先处理缺失值再去除重复行，可能因为缺失值的不同填充结果而漏掉某些重复行。顺序颠倒可能导致级联错误。

## 代码结构

| 清洗步骤 | 处理的问题 | 使用的方法 |
|----------|-----------|-----------|
| 去重 | 重复行 | duplicated() + drop_duplicates() |
| 格式清理 | 前后空格、空字符串 | str.strip() + 替换 |
| 异常值检测（IQR） | 数值超出合理范围 | 四分位距法 |
| 异常值检测（Z-score） | 极端偏离均值 | scipy.stats.zscore |
| 范围修正 | 负数年龄、负数收入 | clip() |
| 类型转换 | 字符串转日期、浮点转整数 | stype() + pd.to_datetime() |

### 1. 构造含脏数据的示例

运行 1. 构造含脏数据的示例 的代码，观察算法在该环节的行为。

In [ ]:
np.random.seed(42)
n = 200
data = pd.DataFrame({
    "ID": list(range(1, n+1)),
    "姓名": np.random.choice(["张三", "李四", "王五", "赵六", "钱七"], n),
# --- 数值计算 ---
    "年龄": np.random.randint(15, 80, n).astype(float),
    "收入": np.random.normal(50000, 15000, n),
    "城市": np.random.choice(["北京", "上海", "广州", "深圳", "杭州"], n),
    "注册日期": pd.date_range("2020-01-01", periods=n, freq="D"),
})

# 注入脏数据
# 1) 重复行
duplicates = data.sample(10, random_state=42)
data = pd.concat([data, duplicates], ignore_index=True)

# 2) 异常值
data.loc[5, "年龄"] = -5          # 负数年龄
data.loc[10, "年龄"] = 200        # 不合理年龄
data.loc[15, "收入"] = -10000     # 负数收入
data.loc[20, "收入"] = 1e7        # 极端高收入

# 3) 空白/特殊字符
data.loc[25, "城市"] = "  北京 "  # 前后空格
data.loc[30, "城市"] = ""         # 空字符串
data.loc[35, "姓名"] = None       # 缺失

print("=== 原始脏数据（前 15 行）===")
print(data.head(15))
print(f"总行数: {len(data)}")

### 2. 处理重复值

运行 2. 处理重复值 的代码，观察算法在该环节的行为。

In [ ]:
dup_count = data.duplicated().sum()
print(f"\n=== 重复行数: {dup_count} ===")
data = data.drop_duplicates()
print(f"去重后行数: {len(data)}")

### 3. 清理空白和特殊字符

运行 3. 清理空白和特殊字符 的代码，观察算法在该环节的行为。

In [ ]:
# 去除字符串列的首尾空白
for col in data.select_dtypes(include="object").columns:
    data[col] = data[col].astype(str).str.strip()
    data[col] = data[col].replace("", np.nan)  # 空字符串转为 NaN

# 替换 "nan" 字符串（由 None 转化而来）
data = data.replace("nan", np.nan)
print(f"\n=== 清理空白后缺失情况 ===")
print(data.isnull().sum())

### 4. 处理异常值（基于 IQR）

运行 4. 处理异常值（基于 IQR） 的代码，观察算法在该环节的行为。

In [ ]:
def detect_outliers_iqr(series, factor=1.5):
    """用 IQR 方法检测异常值，返回布尔掩码"""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
# --- 赋值/配置 ---
    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    return (series < lower) | (series > upper), lower, upper

for col in ["年龄", "收入"]:
    mask, lower, upper = detect_outliers_iqr(data[col].dropna())
    outlier_count = mask.sum()
    print(f"\n{col}: IQR 范围 [{lower:.0f}, {upper:.0f}], 异常值 {outlier_count} 个")

    # 截断法（Winsorize）：将异常值限制在边界内
    data[col] = data[col].clip(lower=lower, upper=upper)

### 5. 基于 Z-score 的异常值检测

运行 5. 基于 Z-score 的异常值检测 的代码，观察算法在该环节的行为。

In [ ]:
from scipy import stats

for col in ["年龄", "收入"]:
    z_scores = np.abs(stats.zscore(data[col].dropna()))
    outlier_z = (z_scores > 3).sum()
    print(f"{col}: Z-score>3 的异常值 {outlier_z} 个")

### 6. 修正数据范围

运行 6. 修正数据范围 的代码，观察算法在该环节的行为。

In [ ]:
# 年龄应在合理范围内
data["年龄"] = data["年龄"].clip(lower=0, upper=120)
# 收入不应为负
data["收入"] = data["收入"].clip(lower=0)
print(f"\n=== 修正后的数据范围 ===")
print(f"年龄: [{data['年龄'].min():.0f}, {data['年龄'].max():.0f}]")
print(f"收入: [{data['收入'].min():.0f}, {data['收入'].max():.0f}]")

### 7. 数据类型转换

运行 7. 数据类型转换 的代码，观察算法在该环节的行为。

In [ ]:
# 将年龄转为整数
data["年龄"] = data["年龄"].astype(int)
# 日期列确保为 datetime 类型
data["注册日期"] = pd.to_datetime(data["注册日期"], errors="coerce")
print(f"\n=== 数据类型 ===")
print(data.dtypes)

### 8. 最终清洗结果

运行 8. 最终清洗结果 的代码，观察算法在该环节的行为。

In [ ]:
print(f"\n=== 清洗后数据（前 10 行）===")
print(data.head(10))
print(f"最终行数: {len(data)}, 缺失数: {data.isnull().sum().sum()}")

print("\n=== 数据清洗流程总结 ===")
print("1. 去除重复行")
print("2. 清理空白字符、统一格式")
print("3. 检测并处理异常值（IQR / Z-score）")
print("4. 修正数据范围（clip 到合理区间）")
# --- 输出结果 ---
print("5. 数据类型转换（确保正确类型）")
print("6. 处理缺失值（参见 缺失值处理.py）")

## 关键代码解释

### 异常值检测：两种流派

**IQR 方法（非参数）：**

```python
def detect_outliers_iqr(series, factor=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    return (series < lower) | (series > upper), lower, upper
```

IQR（四分位距）= Q3 - Q1。低于 Q1 - 1.5×IQR 或高于 Q3 + 1.5×IQR 的值被视为异常值。这是**箱线图**背后的数学原理。
actor=1.5 是经验默认值，可以根据业务需求调整。

**Z-score 方法（参数）：**

```python
z_scores = np.abs(stats.zscore(data[col].dropna()))
outlier_z = (z_scores > 3).sum()
```

假设数据服从正态分布，Z-score > 3 意味着该值落在均值的 3 个标准差之外，概率约为 0.3%。**注意**：Z-score 对非正态分布数据可能失效——一个严重偏态的分布中，大量正常值也可能有高 Z-score。

### 截断法（Winsorize）—— 比删除更温和

```python
data[col] = data[col].clip(lower=lower, upper=upper)
```

clip() 将异常值"截断"到边界值。比如收入的上界是 80000，那么一个 100000 的异常值会被改为 80000。这比直接删除更温和，保留了样本量。

### 字符串清理 —— 看不见的陷阱

```python
for col in data.select_dtypes(include="object").columns:
    data[col] = data[col].astype(str).str.strip()
    data[col] = data[col].replace("", np.nan)
```

"北京" 和 "北京 "（有尾部空格）在计算机看来是两个不同的值。如果不做 strip，后续的分组、合并、编码都会出问题。空字符串也需要转为 NaN，否则会被当作有效类别。

## 使用示例

```python
import pandas as pd
import numpy as np

# 快速检查数据质量
print(df.duplicated().sum())          # 重复行数
print(df.isnull().sum())              # 各列缺失数
print(df.describe())                  # 数值列统计摘要

# 快速异常值检测
q1, q3 = df["收入"].quantile([0.25, 0.75])
iqr = q3 - q1
outliers = df[(df["收入"] < q1 - 1.5*iqr) | (df["收入"] > q3 + 1.5*iqr)]
```

## 注意事项

1. **先观察再清洗**：不要盲目删除异常值，先理解它的来源——是录入错误还是真实极端值？
2. **IQR vs Z-score**：IQR 对偏态分布更鲁棒，Z-score 假设正态分布。实际中 IQR 更常用
3. **业务规则优先**：技术手段（IQR/Z-score）是辅助，业务规则（年龄 0-120、收入 >= 0）才是根本
4. **清洗顺序**：先去重 → 格式清理 → 异常值 → 缺失值。顺序颠倒可能引入新问题
5. **保留原始数据**：清洗过程应该可追溯，建议保存一份清洗前的数据备份

## 总结与延伸

以上代码展示了 **数据清洗** 的完整流程。

**解读要点**：
- 观察处理前后数据的 **统计分布变化**
- 关注缺失值填充策略对下游模型的影响
- 对比不同缩放方法的适用场景

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Great Expectations**：一个 Python 库，用于定义和验证数据质量的"期望"（如"年龄列不应有负值"），适合数据管道中的自动化质量检查
- **数据剖析（Data Profiling）**：pandas-profiling（现名 ydata-profiling）可以一键生成数据质量报告
- **模糊匹配去重**：当重复记录不是完全一致时（如"张三" vs "张 三"），需要模糊字符串匹配（如 
uzzywuzzy 库）
- **自动化清洗管道**：将清洗步骤封装为 sklearn Transformer，可以用 Pipeline 串联，确保训练和推理时执行完全相同的清洗逻辑
